In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
print("Libraries loaded")

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)

# Warm up camera
for i in range(30):
    cap.read()

ret, frame = cap.read()

if ret:
    resized = cv2.resize(frame, (224, 224))
    normalized = resized.astype(np.float32) / 255.0
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    ax1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax1.set_title(f"Raw Camera Feed {frame.shape}")
    ax1.axis('off')
    
    ax2.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
    ax2.set_title("Preprocessed 224x224 - DPU Ready")
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f"Frame shape: {frame.shape}")
    print(f"Normalized shape: {normalized.shape}")
    print(f"Value range: {normalized.min():.2f} - {normalized.max():.2f}")

cap.release()

In [ ]:
import numpy as np

# Use numpy array to simulate shared memory for demo
# (Will use pynq.allocate once DPU bitstream is loaded)
shared_buf = np.zeros((224, 224, 3), dtype=np.float32)
shared_buf[:] = normalized

print("Frame loaded into buffer")
print(f"Buffer shape: {shared_buf.shape}")
print(f"Buffer dtype: {shared_buf.dtype}")
print(f"Value range: {shared_buf.min():.2f} - {shared_buf.max():.2f}")
print("Note: Will use pynq.allocate() for DMA-accessible shared memory once DPU bitstream is loaded")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Read back from shared memory buffer
frame_from_memory = np.array(shared_buf)

# Denormalize for display (convert back from 0-1 to 0-255)
display_frame = (frame_from_memory * 255).astype(np.uint8)

# Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
ax1.set_title("Frame Read Back from Shared Memory")
ax1.axis('off')

ax2.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
ax2.set_title("Original Preprocessed Frame")
ax2.axis('off')

plt.suptitle("Verifying Shared Memory Read/Write", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Shape read from memory: {frame_from_memory.shape}")
print(f"Value range: {frame_from_memory.min():.2f} - {frame_from_memory.max():.2f}")
print("PS successfully wrote and read back from shared memory")
print("Once DPU bitstream loaded, pynq.allocate() gives real physical DDR address")
print("DPU on PL side will read directly from that physical address")

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
for i in range(30):
    cap.read()

print("Streaming... press stop button to end")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
frame_count = 0
start_time = time.time()

try:
    while True:
        ret, frame = cap.read()
        if ret:
            resized = cv2.resize(frame, (224, 224))
            normalized = resized.astype(np.float32) / 255.0
            
            # Update shared memory
            shared_buf[:] = normalized
            
            frame_count += 1
            fps = frame_count / (time.time() - start_time)
            
            ax1.clear()
            ax2.clear()
            
            ax1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax1.set_title(f"Raw Feed - {fps:.1f} FPS")
            ax1.axis('off')
            
            ax2.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
            ax2.set_title("224x224 → Shared Memory → DPU Ready")
            ax2.axis('off')
            
            clear_output(wait=True)
            display(fig)

except KeyboardInterrupt:
    print(f"Stopped - Average FPS: {fps:.1f}")
    cap.release()
    plt.close()

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
from PIL import Image
import io
import base64

cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
for i in range(30):
    cap.read()

# Set lower resolution for speed
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

image_widget = widgets.Image(format='jpeg')
label_widget = widgets.Label()
display(widgets.VBox([image_widget, label_widget]))

frame_count = 0
start_time = time.time()

try:
    while True:
        ret, frame = cap.read()
        if ret:
            resized = cv2.resize(frame, (224, 224))
            
            # Convert to JPEG for fast display
            _, jpeg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
            image_widget.value = jpeg.tobytes()
            
            frame_count += 1
            fps = frame_count / (time.time() - start_time)
            label_widget.value = f"FPS: {fps:.1f} | DPU Ready: {resized.shape}"

except KeyboardInterrupt:
    print(f"Stopped - Average FPS: {fps:.1f}")
    cap.release()

In [ ]:
import tensorflow

In [ ]:
import tflite_runtime.interpreter as tflite

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
import tflite_runtime.interpreter as tflite

# ── Keypoint indices ─────────────────────────────────────────────────────────
NOSE, L_EYE, R_EYE, L_EAR, R_EAR = 0, 1, 2, 3, 4
L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW, R_ELBOW       = 7, 8
L_WRIST, R_WRIST       = 9, 10
L_HIP, R_HIP           = 11, 12
L_KNEE, R_KNEE         = 13, 14
L_ANKLE, R_ANKLE       = 15, 16

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

EDGE_COLORS = {
    (0,1):(255,200,0),(0,2):(255,200,0),(1,3):(255,200,0),(2,4):(255,200,0),
    (5,6):(0,255,255),
    (5,7):(0,255,0),(7,9):(0,255,0),
    (6,8):(255,100,0),(8,10):(255,100,0),
    (5,11):(200,0,255),(6,12):(200,0,255),(11,12):(200,0,255),
    (11,13):(0,100,255),(13,15):(0,100,255),
    (12,14):(0,200,255),(14,16):(0,200,255),
}

# ── Angle utility ────────────────────────────────────────────────────────────
def angle_between(a, b, c):
    """Angle at point b formed by a-b-c. Points are (x, y) tuples."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

def kp_xy(keypoints, idx, w, h):
    """Return (x, y) pixel coords for a keypoint."""
    y, x, _ = keypoints[idx]
    return (x * w, y * h)

def kp_conf(keypoints, *indices):
    """Return True if all keypoints exceed confidence threshold."""
    return all(keypoints[i][2] >= CONF_THRESH for i in indices)

# ── Exercise counters ─────────────────────────────────────────────────────────
class ExerciseCounter:
    def __init__(self, name, color):
        self.name   = name
        self.color  = color
        self.count  = 0
        self.state  = "up"      # "up" or "down"
        self.angle  = 0.0
        self.active = False     # keypoints visible enough to track

    def reset(self):
        self.count = 0
        self.state = "up"

class WorkoutTracker:
    def __init__(self):
        self.squat = ExerciseCounter("Squats",  (0, 200, 255))
        self.curl  = ExerciseCounter("Curls",   (0, 255, 100))
        self.pushup= ExerciseCounter("Pushups", (255, 100, 0))

    def update(self, keypoints, w, h):
        self._update_squat(keypoints, w, h)
        self._update_curl(keypoints, w, h)
        self._update_pushup(keypoints, w, h)

    # ── Squat: knee angle ───────────────────────────────────────────────────
    # Standing: knee ~170°   Bottom: knee ~90°
    def _update_squat(self, kps, w, h):
        if not kp_conf(kps, L_HIP, L_KNEE, L_ANKLE, R_HIP, R_KNEE, R_ANKLE):
            self.squat.active = False
            return
        self.squat.active = True

        l_ang = angle_between(kp_xy(kps, L_HIP,   w, h),
                              kp_xy(kps, L_KNEE,  w, h),
                              kp_xy(kps, L_ANKLE, w, h))
        r_ang = angle_between(kp_xy(kps, R_HIP,   w, h),
                              kp_xy(kps, R_KNEE,  w, h),
                              kp_xy(kps, R_ANKLE, w, h))
        ang = (l_ang + r_ang) / 2
        self.squat.angle = ang

        if ang < 100 and self.squat.state == "up":
            self.squat.state = "down"
        elif ang > 160 and self.squat.state == "down":
            self.squat.state = "up"
            self.squat.count += 1

    # ── Curl: elbow angle ───────────────────────────────────────────────────
    # Extended: elbow ~160°   Curled: elbow ~40°
    def _update_curl(self, kps, w, h):
        if not kp_conf(kps, L_SHOULDER, L_ELBOW, L_WRIST,
                            R_SHOULDER, R_ELBOW, R_WRIST):
            self.curl.active = False
            return
        self.curl.active = True

        l_ang = angle_between(kp_xy(kps, L_SHOULDER, w, h),
                              kp_xy(kps, L_ELBOW,    w, h),
                              kp_xy(kps, L_WRIST,    w, h))
        r_ang = angle_between(kp_xy(kps, R_SHOULDER, w, h),
                              kp_xy(kps, R_ELBOW,    w, h),
                              kp_xy(kps, R_WRIST,    w, h))
        ang = (l_ang + r_ang) / 2
        self.curl.angle = ang

        if ang < 50 and self.curl.state == "up":
            self.curl.state = "down"
        elif ang > 150 and self.curl.state == "down":
            self.curl.state = "up"
            self.curl.count += 1

    # ── Pushup: elbow angle + body horizontal ───────────────────────────────
    # Up: elbow ~160°   Down: elbow ~90°
    def _update_pushup(self, kps, w, h):
        if not kp_conf(kps, L_SHOULDER, L_ELBOW, L_WRIST,
                            R_SHOULDER, R_ELBOW, R_WRIST,
                            L_HIP, R_HIP):
            self.pushup.active = False
            return

        # Check body is roughly horizontal (pushup position)
        # Shoulder Y and hip Y should be close
        l_sh_y = kps[L_SHOULDER][0]
        l_hi_y = kps[L_HIP][0]
        body_angle = abs(l_sh_y - l_hi_y) * h
        if body_angle > 120:   # too vertical — likely standing, not pushup
            self.pushup.active = False
            return

        self.pushup.active = True
        l_ang = angle_between(kp_xy(kps, L_SHOULDER, w, h),
                              kp_xy(kps, L_ELBOW,    w, h),
                              kp_xy(kps, L_WRIST,    w, h))
        r_ang = angle_between(kp_xy(kps, R_SHOULDER, w, h),
                              kp_xy(kps, R_ELBOW,    w, h),
                              kp_xy(kps, R_WRIST,    w, h))
        ang = (l_ang + r_ang) / 2
        self.pushup.angle = ang

        if ang < 90 and self.pushup.state == "up":
            self.pushup.state = "down"
        elif ang > 150 and self.pushup.state == "down":
            self.pushup.state = "up"
            self.pushup.count += 1

# ── Draw helpers ──────────────────────────────────────────────────────────────
def draw_pose(frame, keypoints, conf_thresh=0.25):
    h, w = frame.shape[:2]
    for (a, b) in SKELETON:
        ya, xa, ca = keypoints[a]
        yb, xb, cb = keypoints[b]
        if ca < conf_thresh or cb < conf_thresh:
            continue
        pt_a  = (int(xa * w), int(ya * h))
        pt_b  = (int(xb * w), int(yb * h))
        color = EDGE_COLORS.get((a, b), (200, 200, 200))
        cv2.line(frame, pt_a, pt_b, color, 2, cv2.LINE_AA)
    for (y, x, conf) in keypoints:
        if conf < conf_thresh:
            continue
        cx, cy = int(x * w), int(y * h)
        cv2.circle(frame, (cx, cy), 5, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), 5, (0,   0,   0),   1,  cv2.LINE_AA)
    return frame

def draw_counters(frame, tracker):
    """Draw exercise counters panel on the right side."""
    h, w = frame.shape[:2]
    panel_x = w - 220

    exercises = [tracker.squat, tracker.curl, tracker.pushup]
    for i, ex in enumerate(exercises):
        y = 40 + i * 90

        # Background box
        alpha_box = frame.copy()
        cv2.rectangle(alpha_box, (panel_x, y - 25), (w - 10, y + 60), (30, 30, 30), -1)
        cv2.addWeighted(alpha_box, 0.6, frame, 0.4, 0, frame)

        # Active indicator dot
        dot_color = ex.color if ex.active else (80, 80, 80)
        cv2.circle(frame, (panel_x + 12, y - 5), 6, dot_color, -1)

        # Exercise name
        cv2.putText(frame, ex.name,
                    (panel_x + 25, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1, cv2.LINE_AA)

        # Rep count (large)
        cv2.putText(frame, str(ex.count),
                    (panel_x + 25, y + 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, ex.color, 2, cv2.LINE_AA)

        # State + angle
        state_str = f"{ex.state}  {ex.angle:.0f}deg" if ex.active else "not detected"
        cv2.putText(frame, state_str,
                    (panel_x + 80, y + 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (160, 160, 160), 1, cv2.LINE_AA)

    return frame

def draw_hud(frame, fps, n_detected):
    cv2.rectangle(frame, (0, 0), (280, 32), (0, 0, 0), -1)
    cv2.putText(frame, f"FPS: {fps:4.1f}  Keypoints: {n_detected}/17",
                (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 120), 1, cv2.LINE_AA)
    return frame

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_PATH = "movenet_lightning.tflite"
interpreter = tflite.Interpreter(model_path=MODEL_PATH, num_threads=4)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
INPUT_SIZE  = 192
CONF_THRESH = 0.25

def run_movenet(frame_bgr):
    rgb     = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (INPUT_SIZE, INPUT_SIZE))
    inp     = np.expand_dims(resized, axis=0).astype(np.uint8)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    return interpreter.get_tensor(output_details[0]['index'])[0, 0]

# ── Camera ────────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
for _ in range(30):
    cap.read()
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)

# ── Widgets ───────────────────────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg')
reset_button = widgets.Button(description='Reset Counts',
                              button_style='warning',
                              layout=widgets.Layout(width='150px'))
label_widget = widgets.Label()
display(widgets.VBox([
    image_widget,
    widgets.HBox([reset_button]),
    label_widget
]))

tracker = WorkoutTracker()

def on_reset(_):
    tracker.squat.reset()
    tracker.curl.reset()
    tracker.pushup.reset()

reset_button.on_click(on_reset)

# ── Main loop ─────────────────────────────────────────────────────────────────
frame_count = 0
fps         = 0.0
start_time  = time.time()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        h, w = frame.shape[:2]

        # Inference
        keypoints  = run_movenet(frame)
        n_detected = int((keypoints[:, 2] >= CONF_THRESH).sum())

        # Update exercise counts
        tracker.update(keypoints, w, h)

        # Draw
        annotated = draw_pose(frame.copy(), keypoints, conf_thresh=CONF_THRESH)
        draw_counters(annotated, tracker)

        frame_count += 1
        fps = frame_count / (time.time() - start_time)
        draw_hud(annotated, fps, n_detected)

        # Display
        _, jpeg = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 75])
        image_widget.value = jpeg.tobytes()
        label_widget.value = (f"FPS: {fps:.1f}  |  "
                              f"Squats: {tracker.squat.count}  "
                              f"Curls: {tracker.curl.count}  "
                              f"Pushups: {tracker.pushup.count}")

except KeyboardInterrupt:
    print(f"Stopped — Average FPS: {fps:.1f}")
    print(f"Final counts — Squats: {tracker.squat.count}  "
          f"Curls: {tracker.curl.count}  Pushups: {tracker.pushup.count}")
    cap.release()

In [ ]:
from pynq_dpu import DpuOverlay

overlay = DpuOverlay("dpu.bit")
print(overlay.ip_dict)        # shows all IPs in the bitstream
print(overlay.dpu.descriptions)  # DPU config details